# Integration Examples with External Tools

This notebook demonstrates how to integrate ws3 with external tools and frameworks:

- **FHOPS**: Forest Harvest Operations for cost curve generation
- **FEMIC**: Forest Ecosystem Management for carbon accounting
- **FreshForge**: Workflow automation and pipeline orchestration
- **SpaDES**: Spatial event-driven simulation (via reticulate)
- **REST API**: Web service endpoints for optimization

**Prerequisites:** Completion of `070_ws3_quickstart_complete_workflow.ipynb`

**Note:** Some integrations require additional packages (fhops, femic, freshforge) which may not be installed in all environments.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
import ws3.opt
from ws3.integration import (
    FHOPSIntegrator,
    FEMICIntegrator,
    FreshForgeIntegrator,
    SpaDESIntegrator,
    RESTAPIServer,
    create_fhops_integrator,
    create_femic_integrator,
    create_freshforge_integrator,
    create_spades_integrator,
    create_rest_api
)

print("Imports complete!")

## 1. FHOPS Integration

**FHOPS** (Forest Harvest Operations) generates dynamic harvest cost curves based on:
- Productivity (site quality, stand density, tree size)
- Distance (to landing, road access)
- Terrain (slope, soil conditions)
- Species (harvesting techniques)

FHOPS fills a critical gap: traditional wood supply models use simplified, static harvest costs. FHOPS generates **dynamic harvest cost curves** that vary by productivity parameters.

**Integration Flow:**
```
Inventory Data → FHOPS Cost Modeling → Harvest Cost Curves → ws3 ForestModel → Optimization
```

In [ ]:
# Create fhops integrator
fhops_integrator = create_fhops_integrator()

print(f"FHOPS available: {fhops_integrator._fhops_available}")

# Generate cost curves for a sample scenario
inventory_sample = pd.DataFrame({
    'dt_code': ['CWHvm1_DWG_1'] * 10,
    'area_ha': [50.0] * 10,
    'age': np.arange(20, 120, 10),
    'volume_m3_ha': np.random.uniform(100, 300, 10)
})

cost_curves = fhops_integrator.generate_cost_curves(
    inventory=inventory_sample,
    species='DWG',
    site_index=1,
    distance_to_landing=100.0,
    slope=10.0
)

print("\nGenerated cost curves:")
print(cost_curves.head(10))

# Plot cost curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cost_curves['age'], cost_curves['volume_m3_ha'], 'b-')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Volume (m³/ha)')
axes[0].set_title('Volume by Age')
axes[0].grid(True, alpha=0.3)

axes[1].plot(cost_curves['age'], cost_curves['cost_per_m3'], 'r-')
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Cost ($/m³)')
axes[1].set_title('Harvest Cost by Age')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. FEMIC Integration

**FEMIC** (Forest Ecosystem Management Integration Component) provides detailed carbon pool accounting and modeling.

**Carbon Pools Tracked:**
- Above-ground biomass
- Below-ground biomass
- Deadwood
- Litter
- Soil organic matter
- Harvested products

FEMIC enables comprehensive carbon accounting for harvest schedules, tracking carbon stocks and fluxes across all major forest carbon pools.

In [ ]:
# Create FEMIC integrator
femic_integrator = create_femic_integrator()

print(f"FEMIC available: {femic_integrator._femic_available}")

# Get carbon pools
carbon_pools = femic_integrator.get_carbon_pools()
print(f"\nCarbon pools tracked by FEMIC:")
for pool in carbon_pools:
    print(f"  - {pool}")

# Create a sample schedule for carbon accounting
sample_schedule = pd.DataFrame({
    'period': [1, 1, 2, 2, 3, 3],
    'dt_code': ['CWHvm1_DWG_1', 'CWHvm1_DWG_2'] * 3,
    'area_ha': [50, 30, 40, 20, 60, 25]
})

# Calculate carbon budget
carbon_budget = femic_integrator.calculate_carbon_budget(
    schedule=sample_schedule,
    landscape=sample_schedule
)

print("\nCarbon Budget:")
for key, value in carbon_budget.items():
    print(f"  {key}: {value:.2f}")

# Export carbon report
femic_integrator.export_carbon_report(carbon_budget, 'carbon_budget.json')
print("\nCarbon report exported to carbon_budget.json")

## 3. FreshForge Integration

**FreshForge** provides pipeline orchestration and task management for complex forest modeling workflows.

FreshForge enables:
- Automated workflow execution
- Task dependency management
- Result tracking and provenance
- Reproducible analysis pipelines

Integration allows ws3 to participate in larger forest management workflows with automated data processing, optimization, and reporting.

In [ ]:
# Create FreshForge integrator
ff_integrator = create_freshforge_integrator()

print(f"FreshForge available: {ff_integrator._freshforge_available}")

# Create an optimization pipeline
pipeline = ff_integrator.run_optimization_pipeline(
    model_path="data/woodstock_model_files_tsa24",
    scenario_name="base",
    objective="maximize_npv",
    output_dir="results"
)

print("\nPipeline configuration:")
print(json.dumps(pipeline, indent=2))

# Export pipeline
ff_integrator.export_pipeline(pipeline, 'optimization_pipeline.json')
print("\nPipeline exported to optimization_pipeline.json")

# Display pipeline steps
print("\nPipeline Steps:")
for step in pipeline['steps']:
    print(f"  {step['id']}: {step['type']}")

## 4. SpaDES Integration

**SpaDES** (SPAtial Event-driven Simulation Engine) is an R framework for building spatially-explicit, event-driven forest landscape simulations.

Integration with ws3 enables:
- Spatially-explicit harvest scheduling
- Event-driven simulation with harvest decisions
- Dynamic landscape modeling
- Integration with disturbance processes (fire, insects, wind)

The `spades_ws3` R module bridges SpaDES and ws3 using `reticulate` (R-Python bridge).

In [ ]:
# Create SpaDES integrator
spades_integrator = create_spades_integrator()

print(f"SpaDES available: {spades_integrator._spades_available}")

# Create SpaDES configuration
spades_config = spades_integrator.create_spades_config(
    model_path="data/woodstock_model_files_tsa24",
    landscape_raster="data/landscape.tif",
    scheduling_mode='optimize'
)

print("\nSpaDES Configuration:")
print(json.dumps(spades_config, indent=2))

# Export configuration
spades_integrator.export_config(spades_config, 'spades_config.json')
print("\nSpaDES configuration exported to spades_config.json")

## 5. REST API Integration

ws3 can expose optimization capabilities as a REST API for:
- Web applications
- Mobile apps
- Integration with other systems
- Batch processing workflows

The REST API provides endpoints for:
- Running optimization scenarios
- Retrieving results
- Health monitoring
- Scenario management

In [ ]:
# Create REST API server
api_server = create_rest_api(host='0.0.0.0', port=8000)

print(f"REST API server created")
print(f"  Host: {api_server.host}")
print(f"  Port: {api_server.port}")

# Create FastAPI app
app = api_server.create_app()

print(f"\nAPI Endpoints:")
print(f"  GET  /           - API info")
print(f"  GET  /health     - Health check")
print(f"  POST /optimize   - Run optimization")
print(f"  GET  /results/{{id}} - Get results")

# Note: To actually run the server, uncomment the following line:
# api_server.run_server(app)
print("\nTo run the API server:")
print("  api_server.run_server(app)")
print("\nThe server will be available at:")
print(f"  http://{api_server.host}:{api_server.port}")

## Summary

**Integration Capabilities:**

| Integration | Purpose | Status |
|------------|---------|--------|
| FHOPS | Harvest cost curves | ✅ Available |
| FEMIC | Carbon accounting | ✅ Available |
| FreshForge | Workflow automation | ✅ Available |
| SpaDES | Spatial simulation | ✅ Available (requires R) |
| REST API | Web service | ✅ Available |

**Key Benefits:**

1. **FHOPS**: Dynamic harvest cost modeling based on productivity
2. **FEMIC**: Comprehensive carbon pool accounting
3. **FreshForge**: Automated pipeline orchestration
4. **SpaDES**: Spatially-explicit event-driven simulation
5. **REST API**: Web service for optimization

**When to Use Each Integration:**

- Use **FHOPS** when you need dynamic harvest costs
- Use **FEMIC** for carbon accounting and climate scenarios
- Use **FreshForge** for complex multi-step workflows
- Use **SpaDES** for spatial simulations with harvest decisions
- Use **REST API** for web applications and batch processing

**Next Steps:**

- Install additional packages as needed: `pip install fhops femic freshforge`
- Configure integrations for your specific use case
- Explore the integration modules for more details